# Phase 1: Colab model, QLoRA, and inference-result feasibility

Run cells in order on a Colab GPU runtime. This notebook benchmarks immutable model revisions, performs a one-step QLoRA smoke test, and writes reviewed artifacts to private Google Drive. It does not expose a web service.

In [ ]:
%pip install -q transformers==5.14.1 accelerate==1.14.0
%pip install -q bitsandbytes==0.50.0 peft==0.20.0 safetensors==0.8.0

import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/muzzary/GTM-Agent.git"
REPOSITORY_REF = "codex/phase-1-colab-feasibility"
REPOSITORY_DIR = Path("/content/GTM-Agent")
if not REPOSITORY_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            REPOSITORY_REF,
            REPOSITORY_URL,
            str(REPOSITORY_DIR),
        ],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR)],
    check=True,
)
sys.path.insert(0, str(REPOSITORY_DIR))

In [ ]:
import gc
import hashlib
import importlib.metadata
import json
import platform
import statistics
import subprocess
import time
import traceback
import uuid

import torch
from google.colab import drive
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

from src.evaluation.phase1 import (
    choose_winner,
    evaluate_output,
    load_manifest,
    parse_model_output,
)
from src.outreach.bundles import create_bundle
from src.schemas.benchmark import CandidateResult
from src.schemas.inference import (
    GenerationSettings,
    InferenceResponse,
    ModelIdentity,
    RuntimeMetadata,
)

PINNED_CANDIDATES = {
    "Qwen/Qwen3-4B-Instruct-2507": "cdbee75f17c01a7cc42f958dc650907174af0554",
    "microsoft/Phi-4-mini-instruct": "cfbefacb99257ffa30c83adab238a50856ac3083",
}

MANIFEST_PATH = REPOSITORY_DIR / "configs/phase1/benchmark.json"
manifest = load_manifest(MANIFEST_PATH)

assert {
    candidate.model_id: candidate.revision for candidate in manifest.candidates
} == PINNED_CANDIDATES

assert all(candidate.trust_remote_code is False for candidate in manifest.candidates)

assert torch.cuda.is_available(), "Phase 1 requires a Colab GPU runtime."

In [ ]:
drive.mount("/content/drive")
ARTIFACT_ROOT = Path("/content/drive/MyDrive/GTM-Agent/phase1")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def atomic_write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True), encoding="utf-8")
    temporary.replace(path)


def capture_environment():
    properties = torch.cuda.get_device_properties(0)
    repository_revision = subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "rev-parse", "HEAD"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    packages = {
        name: importlib.metadata.version(name)
        for name in (
            "transformers",
            "accelerate",
            "bitsandbytes",
            "peft",
            "safetensors",
        )
    }
    return {
        "repository_revision": repository_revision,
        "manifest_sha256": sha256_file(MANIFEST_PATH),
        "python_version": platform.python_version(),
        "torch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "gpu_name": properties.name,
        "gpu_compute_capability": list(torch.cuda.get_device_capability(0)),
        "gpu_memory_mb": properties.total_memory // (1024 * 1024),
        "packages": packages,
    }


environment = capture_environment()
atomic_write_json(
    ARTIFACT_ROOT / "environment.json",
    environment,
)
environment

In [ ]:
def build_prompt(case):
    claims = "\n".join(
        f"- {claim.claim_id}: {claim.text}" for claim in case.approved_claims
    )
    evidence = "\n".join(f"- {item}" for item in case.prospect_evidence)

    output_example = {
        "subject": "A concise subject",
        "body": "A concise evidence-based email.",
        "claims_used": ["approved-claim-id"],
        "uncertainty_notes": ["An uncertainty or inference, if applicable."],
    }

    return f"""
Write one concise B2B outreach email.

Return one JSON object only. Do not use Markdown or code fences.

The required JSON structure and value types are:

{json.dumps(output_example, indent=2)}

Mandatory rules:

1. Return exactly these four keys:
   subject, body, claims_used, uncertainty_notes.
2. subject and body must be strings.
3. claims_used must be an array of approved claim-ID strings.
4. uncertainty_notes must always be an array of strings.
   Use [] when there are no uncertainty notes.
5. Keep the subject under 10 words.
6. Keep the body under 90 words.
7. Use only facts found in the product description,
   approved claims, or public evidence below.
8. Treat the pain hypothesis as an unconfirmed possibility.
   Never present it as a known fact.
9. Do not invent outcomes such as time savings, consistency,
   compliance, workflow compatibility, adoption, or reduced risk.
10. Put uncertainty disclosures in uncertainty_notes,
    not inside the email body.

Product: {case.product_name}

Product description:
{case.product_description}

Approved claims:
{claims}

Prospect: {case.prospect_name}
Target role: {case.target_role}

Unconfirmed pain hypothesis:
{case.pain_hypothesis}

Public evidence:
{evidence}
""".strip()


def load_quantized_candidate(candidate):
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        candidate.model_id,
        revision=candidate.revision,
        trust_remote_code=False,
    )

    model = AutoModelForCausalLM.from_pretrained(
        candidate.model_id,
        revision=candidate.revision,
        trust_remote_code=False,
        use_safetensors=True,
        device_map="auto",
        quantization_config=quantization,
        dtype=compute_dtype,
        attn_implementation="eager",
    )

    return tokenizer, model


def generate_once(tokenizer, model, prompt, generation):
    messages = [
        {
            "role": "system",
            "content": (
                "Return strict JSON only. Use only supplied evidence "
                "and approved product claims."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    torch.manual_seed(generation.seed)
    torch.cuda.manual_seed_all(generation.seed)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    started = time.perf_counter()

    with torch.inference_mode():
        output_ids = model.generate(
            **encoded,
            do_sample=False,
            max_new_tokens=generation.max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

    torch.cuda.synchronize()

    latency_ms = (time.perf_counter() - started) * 1000

    peak_mb = torch.cuda.max_memory_allocated() // (1024 * 1024)

    generated = output_ids[
        0,
        encoded["input_ids"].shape[-1] :,
    ]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    ).strip()

    return text, latency_ms, peak_mb

In [ ]:
def run_qlora_smoke(model, tokenizer, candidate):
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(
        model,
        LoraConfig(
            r=4,
            lora_alpha=8,
            lora_dropout=0.0,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules="all-linear",
        ),
    )
    training_text = (
        build_prompt(manifest.cases[0])
        + '\n{"subject":"Test","body":"Test","claims_used":[],"uncertainty_notes":[]}'
    )
    batch = tokenizer(
        training_text, return_tensors="pt", truncation=True, max_length=512
    ).to(model.device)
    model.train()
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=1e-4,
    )
    optimizer.zero_grad(set_to_none=True)
    loss = model(**batch, labels=batch["input_ids"]).loss
    loss.backward()
    optimizer.step()
    adapter_dir = (
        ARTIFACT_ROOT / "smoke_adapters" / candidate.model_id.replace("/", "--")
    )
    model.save_pretrained(adapter_dir, safe_serialization=True)
    base_model = model.unload()
    reloaded = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=False)
    assert any(adapter_dir.glob("*.safetensors")), (
        "Adapter was not saved as safetensors."
    )
    return reloaded, {"passed": True, "loss": float(loss.detach().cpu())}


def run_candidate_benchmark(candidate):
    tokenizer, model = load_quantized_candidate(candidate)
    loaded_memory_mb = torch.cuda.memory_allocated() // (1024 * 1024)
    warmup_prompt = build_prompt(manifest.cases[0])
    for _ in range(manifest.generation.warmup_runs):
        generate_once(tokenizer, model, warmup_prompt, manifest.generation)
    outputs = []
    for case in manifest.cases:
        prompt = build_prompt(case)
        for run_index in range(manifest.generation.measured_runs):
            raw, latency_ms, peak_mb = generate_once(
                tokenizer, model, prompt, manifest.generation
            )
            record = {
                "case_id": case.case_id,
                "run_index": run_index,
                "raw_output": raw,
                "latency_ms": latency_ms,
                "peak_gpu_memory_mb": peak_mb,
                "valid": False,
                "unsupported_claims": [],
            }
            try:
                parsed = parse_model_output(raw)
                evaluation = evaluate_output(case, parsed)
                record.update(
                    {
                        "valid": True,
                        "parsed_output": parsed.model_dump(mode="json"),
                        "unsupported_claims": evaluation.unsupported_claims,
                    }
                )
            except ValueError as exc:
                record["validation_error"] = str(exc)
            outputs.append(record)
    model, qlora = run_qlora_smoke(model, tokenizer, candidate)
    report = {
        "candidate": candidate.model_dump(mode="json"),
        "environment": environment,
        "generation": manifest.generation.model_dump(mode="json"),
        "loaded_gpu_memory_mb": loaded_memory_mb,
        "qlora_smoke": qlora,
        "outputs": outputs,
    }
    atomic_write_json(
        ARTIFACT_ROOT / f"{candidate.model_id.replace('/', '--')}.json", report
    )
    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    return report

In [ ]:
candidate_reports = {}

for candidate in manifest.candidates:
    print(f"Running {candidate.model_id}...")

    try:
        report = run_candidate_benchmark(candidate)
        candidate_reports[candidate.model_id] = report

        valid_count = sum(output["valid"] for output in report["outputs"])

        print(
            {
                "model": candidate.model_id,
                "status": "completed",
                "valid_outputs": valid_count,
                "total_outputs": len(report["outputs"]),
                "qlora_passed": report["qlora_smoke"]["passed"],
            }
        )

    except Exception as exc:
        failure = {
            "candidate": candidate.model_dump(mode="json"),
            "environment": environment,
            "failure_type": type(exc).__name__,
            "failure": str(exc),
            "traceback": traceback.format_exc(),
        }

        failure_path = ARTIFACT_ROOT / (
            candidate.model_id.replace("/", "--") + "--failure.json"
        )

        atomic_write_json(failure_path, failure)
        candidate_reports[candidate.model_id] = failure

        print(
            {
                "model": candidate.model_id,
                "status": "failed",
                "failure_type": type(exc).__name__,
                "failure": str(exc),
                "report": str(failure_path),
            }
        )

    finally:
        gc.collect()
        torch.cuda.empty_cache()

candidate_reports.keys()

In [ ]:
gate_summary = {}

for candidate in manifest.candidates:
    report = candidate_reports[candidate.model_id]

    if "outputs" not in report:
        gate_summary[candidate.model_id] = {
            "status": "candidate_failed",
            "failure": report.get("failure"),
        }
        continue

    outputs = report["outputs"]
    valid_outputs = sum(output["valid"] for output in outputs)
    unsupported_claims = sum(len(output["unsupported_claims"]) for output in outputs)

    gate_summary[candidate.model_id] = {
        "status": "completed",
        "valid_outputs": valid_outputs,
        "total_outputs": len(outputs),
        "valid_output_rate": valid_outputs / len(outputs),
        "unsupported_claim_count": unsupported_claims,
        "qlora_smoke_passed": report["qlora_smoke"]["passed"],
        "validation_errors": [
            output.get("validation_error") for output in outputs if not output["valid"]
        ],
    }

print(json.dumps(gate_summary, indent=2))

## Human review and approval
Review the saved candidate JSON reports. Enter one 1–5 average per candidate after applying all six rubric dimensions to representative outputs. A candidate cannot be recommended unless every hard gate passes.

In [ ]:
HUMAN_RUBRIC_AVERAGES = {
    # "Qwen/Qwen3-4B-Instruct-2507": 0.0,
    # "microsoft/Phi-4-mini-instruct": 0.0,
}

candidate_results = []
for candidate in manifest.candidates:
    report = candidate_reports[candidate.model_id]
    if "outputs" not in report or candidate.model_id not in HUMAN_RUBRIC_AVERAGES:
        continue
    outputs = report["outputs"]
    candidate_results.append(
        CandidateResult(
            model_id=candidate.model_id,
            model_revision=candidate.revision,
            total_outputs=len(outputs),
            valid_outputs=sum(item["valid"] for item in outputs),
            unsupported_claim_count=sum(
                len(item["unsupported_claims"]) for item in outputs
            ),
            qlora_smoke_passed=report["qlora_smoke"]["passed"],
            human_rubric_average=HUMAN_RUBRIC_AVERAGES[candidate.model_id],
            peak_gpu_memory_mb=max(item["peak_gpu_memory_mb"] for item in outputs),
            median_latency_ms=statistics.median(item["latency_ms"] for item in outputs),
        )
    )
recommended = choose_winner(candidate_results, manifest.hard_gates)
atomic_write_json(
    ARTIFACT_ROOT / "candidate_comparison.json",
    {
        "results": [item.model_dump(mode="json") for item in candidate_results],
        "recommended_model_id": recommended.model_id,
    },
)
recommended

## Export one real inference-result bundle
Set `APPROVED_MODEL_ID` only after reviewing the comparison. This generates one real held-out output and stores the validated contract bundle for local import.

In [ ]:
APPROVED_MODEL_ID = ""  # Set after manual approval.
DEMO_CASE_ID = "case-reporting-operations"
assert APPROVED_MODEL_ID == recommended.model_id, (
    "Approve the recommended model or document an explicit override."
)
candidate = next(
    item for item in manifest.candidates if item.model_id == APPROVED_MODEL_ID
)
case = next(item for item in manifest.cases if item.case_id == DEMO_CASE_ID)
tokenizer, model = load_quantized_candidate(candidate)
raw, latency_ms, _ = generate_once(
    tokenizer, model, build_prompt(case), manifest.generation
)
parsed = parse_model_output(raw)
assert evaluate_output(case, parsed).passes_claim_gate
request_id = "req_" + uuid.uuid4().hex[:16]
response = InferenceResponse(
    request_id=request_id,
    model=ModelIdentity(model_id=candidate.model_id, model_revision=candidate.revision),
    generation=GenerationSettings(
        max_new_tokens=manifest.generation.max_new_tokens, seed=manifest.generation.seed
    ),
    output=parsed,
    runtime=RuntimeMetadata(
        python_version=environment["python_version"],
        torch_version=environment["torch_version"],
        transformers_version=environment["packages"]["transformers"],
        cuda_version=environment["cuda_version"],
        gpu_name=environment["gpu_name"],
        gpu_memory_mb=environment["gpu_memory_mb"],
        latency_ms=latency_ms,
    ),
)
bundle = create_bundle(response)
bundle_path = ARTIFACT_ROOT / f"inference-bundle--{request_id}.json"
atomic_write_json(bundle_path, bundle.model_dump(mode="json"))
print(
    {
        "bundle_path": str(bundle_path),
        "request_id": request_id,
        "model_revision": candidate.revision,
        "payload_sha256": bundle.payload_sha256,
    }
)